Projeto: Merca Data Platform

Squad: 2 | Streaming em Tempo Real

| **Objetivo** | Polling contínuo de novos snapshots no ADLS |

| **Modo atual** | Infinito (substituir por near real-time quando disponível) |

| **Intervalo** | 30 segundos entre verificações |

| **Depende de** | feat_squad2_99_helpers |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
# ─────────────────────────────────────────────
# CONFIGURAÇÕES DO POLLING
# TODO: Substituir POLLING_INFINITO = True por
#       agendamento near real-time quando
#       infraestrutura estiver disponível
# ─────────────────────────────────────────────

POLLING_INFINITO  = True   # False = usa MAX_CICLOS
MAX_CICLOS        = 10     # usado apenas se POLLING_INFINITO = False
INTERVALO_SEG     = 30     # segundos entre verificações
TABELAS_MONITORAR = TABELAS_SQUAD2

log.info("Configurações do Polling:")
log.info(f"  Modo     : {'Infinito' if POLLING_INFINITO else f'Limitado ({MAX_CICLOS} ciclos)'}")
log.info(f"  Intervalo: {INTERVALO_SEG}s")
log.info(f"  Tabelas  : {TABELAS_MONITORAR}")


In [0]:
# ─────────────────────────────────────────────
# CONFIGURAÇÕES DO POLLING
# TODO: Substituir POLLING_INFINITO = True por
#       agendamento near real-time quando
#       infraestrutura estiver disponível
# ─────────────────────────────────────────────

POLLING_INFINITO  = False   # True quando for para produção
MAX_CICLOS        = 10     # usado apenas se POLLING_INFINITO = False
INTERVALO_SEG     = 30     # segundos entre verificações
TABELAS_MONITORAR = TABELAS_SQUAD2

log.info("Configurações do Polling:")
log.info(f"  Modo     : {'Infinito' if POLLING_INFINITO else f'Limitado ({MAX_CICLOS} ciclos)'}")
log.info(f"  Intervalo: {INTERVALO_SEG}s")
log.info(f"  Tabelas  : {TABELAS_MONITORAR}")

In [0]:
# ─────────────────────────────────────────────
# FUNÇÕES DO POLLING
# ─────────────────────────────────────────────

def processar_snapshot(snapshot_id: str) -> dict:
    """
    Processa todas as tabelas de um snapshot novo.
    Lê os parquet e grava no SQL Server.

    Args:
        snapshot_id: caminho do snapshot ex: 2026/04/27/225222

    Returns:
        dict: resultado do processamento por tabela
    """
    resultado = {}

    for tabela in TABELAS_MONITORAR:
        try:
            # Lê do ADLS
            df = ler_parquet(snapshot_id, tabela)

            # Grava no SQL Server
            sucesso = gravar_sql(df, tabela, mode="append")

            resultado[tabela] = {
                "status" : "OK" if sucesso else "ERRO",
                "linhas" : df.count()
            }

        except Exception as e:
            log.error(f"Erro ao processar {tabela}: {str(e)}")
            resultado[tabela] = {
                "status": "ERRO",
                "erro"  : str(e)
            }

    return resultado


def log_resultado(snapshot_id: str, resultado: dict) -> None:
    """Loga o resultado do processamento de um snapshot."""
    log.info(f"Resultado snapshot {snapshot_id}:")
    for tabela, info in resultado.items():
        if info["status"] == "OK":
            log.info(f"  OK {tabela} → {info['linhas']} linhas")
        else:
            log.error(f"  ERRO {tabela} → {info.get('erro', 'erro desconhecido')}")


In [0]:
# ─────────────────────────────────────────────
# LOOP DE POLLING
# ─────────────────────────────────────────────

inicio = log_inicio("feat_squad2_01_polling_ingestao")

# Carrega snapshots existentes como já processados
snapshots_processados = listar_snapshots()
total_processados     = 0
ciclo                 = 0

log.info(f"{len(snapshots_processados)} snapshot(s) existentes ignorados.")
log.info("Aguardando novos snapshots...\n")

try:
    while POLLING_INFINITO or ciclo < MAX_CICLOS:
        ciclo += 1
        agora  = datetime.now().strftime("%H:%M:%S")

        log.info(f"[{agora}] Ciclo {ciclo} — verificando novos snapshots...")

        try:
            # Lista snapshots atuais
            snapshots_atuais = listar_snapshots()

            # Detecta novos
            novos = snapshots_atuais - snapshots_processados

            if novos:
                log.info(f"  {len(novos)} novo(s) snapshot(s) detectado(s)!")

                for snapshot_id in sorted(novos):
                    resultado = processar_snapshot(snapshot_id)
                    log_resultado(snapshot_id, resultado)
                    snapshots_processados.add(snapshot_id)
                    total_processados += 1

            else:
                log.info(" Nenhum snapshot novo encontrado.")

        except Exception as e:
            # Erro em um ciclo não interrompe o polling
            log.error(f"Erro no ciclo {ciclo}: {str(e)}")
            log.info("Continuando polling...")

        # Aguarda antes do próximo ciclo
        log.info(f"Aguardando {INTERVALO_SEG}s...")
        time.sleep(INTERVALO_SEG)

except KeyboardInterrupt:
    # Permite interrupção manual sem erro
    log.info("Polling interrompido manualmente.")

finally:
    log.info(f"Total de snapshots processados: {total_processados}")
    log_fim("feat_squad2_01_polling_ingestao", inicio)